# TP Jour 1 — Choix du Sujet Kaggle, Stockage Distribué HDFS & Couche Bronze

Bienvenue dans ce fil rouge **Data Lakehouse** ! Vous êtes libres de choisir votre thématique métier et votre jeu de données public sur **Kaggle**.

### Objectifs de la séance
1. **Cadrer votre sujet métier** et sélectionner un dataset Kaggle adapté au calcul distribué (~50 à 500 Mo).
2. **Explorer le cluster HDFS réel** (`1 NameNode` + `3 DataNodes`) configuré avec `dfs.blocksize = 16 Mo` et `dfs.replication = 2`.
3. **Ingérer vos fichiers bruts sur HDFS** et observer physiquement leur découpage en blocs de 16 Mo.
4. **Réaliser un crash-test en direct** : couper `datanode2` et observer la détection de panne + l'auto-réparation vers le 3e DataNode en ~50 secondes.
5. **Construire votre couche Bronze** (`/lakehouse/bronze/`) au format **Delta Lake** et inspecter votre premier job sur **Spark Web UI** (`http://localhost:4040`).

---
### Fiche de cadrage de notre projet : Projet Data Lakehouse Football
- **Titre du sujet :** `Analyse des performances et des résultats du football professionnel européen et international`
- **Identifiant Kaggle (`auteur/nom-du-dataset`) :** `adamgbor/club-football-match-data-2000-2025`
- **Problématique métier principale :** `Comment exploiter un historique de plusieurs centaines de milliers de matchs de football afin d'analyser les performances des équipes, comparer les championnats, identifier les facteurs influençant les résultats et étudier l'évolution du niveau des clubs au fil des saisons ?`
- **Variables clés retenues (temporelles, quantitatives, catégorielles) :** `Temporelles : date du match, saison, période. Quantitatives : but marquées, tir, corners, fautes, carton. Catégorielles : équipe à domicile, équipe à l'extérieur, championnat, pays. `

## 1. Téléchargement de votre Dataset Kaggle dans `/workspace/data_raw`

Renseignez dans la cellule suivante l'identifiant Kaggle de votre dataset (ex. `"auteur/mon-dataset"`), ou déposez directement vos fichiers dans le dossier `data_raw/` à la racine du projet.
> **Critère important :** Vérifiez qu'au moins un fichier dépasse **20 à 32 Mo** afin d'observer sa division en plusieurs blocs HDFS de **16 Mo** !

In [3]:
import shutil
from pathlib import Path


DATA_RAW = Path("/workspace/data_raw")
DATA_RAW.mkdir(parents=True, exist_ok=True)
SUPPORTED_EXTS = {".csv", ".parquet", ".json", ".jsonl", ".tsv"}

# TODO : Renseignez l'identifiant Kaggle de votre sujet (format "auteur/nom-du-dataset")
KAGGLE_DATASET = "adamgbor/club-football-match-data-2000-2025"  # Exemple : "auteur/nom-du-dataset"

local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

if not local_files:
    assert KAGGLE_DATASET.strip(), (
        "TODO : Renseignez `KAGGLE_DATASET = ''` (ou déposez vos fichiers dans data_raw/)."
    )
    import kagglehub
    print(f"Téléchargement de {KAGGLE_DATASET} depuis Kaggle Hub...")
    cache_dir = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    for src_file in sorted(cache_dir.rglob("*")):
        if src_file.is_file() and src_file.suffix.lower() in SUPPORTED_EXTS:
            shutil.copy2(src_file, DATA_RAW / src_file.name)
    local_files = sorted([f for f in DATA_RAW.iterdir() if f.is_file() and f.suffix.lower() in SUPPORTED_EXTS])

assert len(local_files) > 0, "Aucun fichier de données détecté dans /workspace/data_raw/."

print(f"Inventaire local dans {DATA_RAW} ({len(local_files)} fichier(s)) :")
for f in local_files:
    size_mb = f.stat().st_size / (1024 * 1024)
    expected_blocks = max(1, int(-(-f.stat().st_size // (16 * 1024 * 1024))))
    print(f" - {f.name:<45} {size_mb:>7.2f} Mo  (~{expected_blocks} bloc(s) HDFS de 16 Mo)")

c:\Users\bapti\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Téléchargement de adamgbor/club-football-match-data-2000-2025 depuis Kaggle Hub...


100%|██████████| 15.2M/15.2M [00:00<00:00, 18.7MB/s]

Extracting files...


Inventaire local dans \workspace\data_raw (2 fichier(s)) :
 - EloRatings.csv                                  10.78 Mo  (~1 bloc(s) HDFS de 16 Mo)
 - Matches.csv                                     43.49 Mo  (~3 bloc(s) HDFS de 16 Mo)


In [4]:
pip install kagglehub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Démarrage de la `SparkSession` pré-configurée sur HDFS

Grâce au fichier `config/spark-defaults.conf`, Spark pointe automatiquement vers `hdfs://namenode:9000` avec **Delta Lake** activé.

### Exercice 2.1 — Initialiser `SparkSession` et contrôler la configuration HDFS
1. Créez une session Spark nommée `"J1_HDFS_Ingestion_Bronze"`.
2. Récupérez via `spark.conf.get(...)` la valeur de `"spark.hadoop.fs.defaultFS"`, `"spark.hadoop.dfs.blocksize"` et `"spark.hadoop.dfs.replication"`.

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# TODO : Initialiser la SparkSession
spark = (
    SparkSession.builder
    .appName("J1_HDFS_Ingestion_Bronze")
    .getOrCreate()
)

assert spark is not None, "TODO : Initialisez `spark` avec SparkSession.builder..."

# TODO : Lire les 3 paramètres HDFS depuis `spark.conf.get(...)`
default_fs = spark.conf.get("spark.hadoop.fs.defaultFS")
block_size = spark.conf.get("spark.hadoop.dfs.blocksize")
replication = spark.conf.get("spark.hadoop.dfs.replication")

print("Spark version       :", spark.version)
print("Default FileSystem  :", default_fs)
print("HDFS Block Size (o) :", block_size)
print("HDFS Replication    :", replication)

Spark version       : 3.5.3
Default FileSystem  : hdfs://namenode:9000
HDFS Block Size (o) : 16777216
HDFS Replication    : 2


## 3. Copie des fichiers bruts vers HDFS & Inspection des Blocs de 16 Mo

La cellule suivante téléverse vos fichiers bruts de `/workspace/data_raw` vers `/lakehouse/bronze/raw/` sur HDFS, puis interroge le NameNode pour afficher la localisation exacte de chaque bloc de 16 Mo sur les 3 DataNodes (`datanode1`, `datanode2`, `datanode3`).

In [9]:

# Connexion à HDFS via Spark

jvm = spark._jvm
hadoop_conf = spark._jsc.hadoopConfiguration()

fs = jvm.org.apache.hadoop.fs.FileSystem.get(hadoop_conf)
PathHDFS = jvm.org.apache.hadoop.fs.Path

# Dossier des fichiers bruts dans HDFS
raw_hdfs_dir = PathHDFS("/lakehouse/bronze/raw")

# Vérification de l'existence du dossier
if not fs.exists(raw_hdfs_dir):
    raise Exception("Le dossier HDFS /lakehouse/bronze/raw est introuvable.")

print("Connexion HDFS réussie !")
print("Dossier :", raw_hdfs_dir.toString())

# Vérification des fichiers déjà présents
print("\nFichiers présents dans HDFS :")

for status in fs.listStatus(raw_hdfs_dir):
    print(
        f"- {status.getPath().getName()} "
        f"({status.getLen() / (1024 * 1024):.2f} Mo)"
    )


Connexion HDFS réussie !
Dossier : /lakehouse/bronze/raw

Fichiers présents dans HDFS :
- EloRatings.csv (10.78 Mo)
- Matches.csv (43.49 Mo)


In [10]:
# Inspection physique du découpage en blocs de 16 Mo et identification du plus gros fichier pour le crash-test
statuses = sorted(fs.listStatus(raw_hdfs_dir), key=lambda s: s.getLen(), reverse=True)
largest_hdfs_path = statuses[0].getPath().toString()

for status in statuses:
    path = status.getPath()
    length_mb = status.getLen() / (1024 * 1024)
    blocks = fs.getFileBlockLocations(status, 0, status.getLen())
    print(f"Fichier : {path.getName()} ({length_mb:.2f} Mo | repl={status.getReplication()})")
    for idx, blk in enumerate(blocks):
        print(f"   Bloc #{idx} ({blk.getLength() / (1024*1024):5.2f} Mo) -> DataNodes : {list(blk.getHosts())}")

Fichier : Matches.csv (43.49 Mo | repl=2)
   Bloc #0 (16.00 Mo) -> DataNodes : ['datanode3', 'datanode1']
   Bloc #1 (16.00 Mo) -> DataNodes : ['datanode1', 'datanode3']
   Bloc #2 (11.49 Mo) -> DataNodes : ['datanode3', 'datanode1']
Fichier : EloRatings.csv (10.78 Mo | repl=2)
   Bloc #0 (10.78 Mo) -> DataNodes : ['datanode3', 'datanode1']


### Questions d'analyse (à vérifier aussi sur http://localhost:9870)
1. Comparez sur l'interface Web du NameNode (`Utilities > Browse the file system`) le nombre de blocs de votre fichier le plus volumineux avec sa taille divisée par 16 Mo.

Pour Matches.csv, le fichier fait 43,49 Mo et possède 3 blocs de 16 Mo maximum.

Calcul : \(43,49 / 16 \approx 2,72\), donc 3 blocs.

2. Un fichier de 3 Mo bloque-t-il réellement 16 Mo d'espace disque sur les DataNodes ? Pourquoi ?

Non. Le dernier bloc d'un fichier peut être plus petit que la taille configurée. Un fichier de 3 Mo occupe donc environ 3 Mo de données par réplique, et non 16 Mo.

3. Pourquoi avoir configuré `dfs.replication = 2` sur un cluster à `3` DataNodes ?

Pour assurer la tolérance aux pannes tout en répartissant les données. Chaque bloc possède deux répliques sur deux DataNodes différents. Si un DataNode tombe en panne, une autre copie reste disponible.

---
## 4. Crash-Test en Direct : Panne d'un DataNode et Auto-Réparation (~50s)

Dans `config/hdfs-site.xml`, le délai de détection d'un DataNode mort a été abaissé de 10m30s à **~50 secondes** :
$$2 \times \text{recheck-interval (10s)} + 10 \times \text{heartbeat.interval (3s)} = 50\text{ s}$$

### Protocole à réaliser depuis votre terminal :
1. Inspectez l'état initial des blocs : `task hdfs:fsck`
2. Arrêtez brutalement le DataNode 2 :
   ```bash
   task hdfs:crash-dn2   # équivalent à : docker stop datanode2
   ```
3. Observez l'onglet **Datanodes** sur `http://localhost:9870` pendant ~50s : dès que `datanode2` passe à **Dead**, le NameNode détecte les blocs sous-répliqués et ordonne leur copie automatique vers le 3e DataNode survivant !

### Exercice 4.1 — Vérifier la continuité de service et la *Data Locality*
Pendant que `datanode2` est éteint, lisez votre plus gros fichier (`largest_hdfs_path`) avec PySpark, affichez `df_check.count()` ainsi que `df_check.rdd.getNumPartitions()`, et expliquez le lien avec le nombre de blocs HDFS.

*(Pensez ensuite à rallumer le nœud avec `task hdfs:restore-dn2` !)*

In [12]:
print("Fichier cible du test de résilience :", largest_hdfs_path)

# TODO : Lire `largest_hdfs_path` avec PySpark (selon son format : spark.read.csv / parquet / json)
df_check = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(largest_hdfs_path)
)

assert df_check is not None, "TODO : Lisez `largest_hdfs_path` dans `df_check`."

print(f"Nombre total de lignes lues       : {df_check.count():,}")
print(f"Nombre de partitions Spark créées : {df_check.rdd.getNumPartitions()}")

Fichier cible du test de résilience : hdfs://namenode:9000/lakehouse/bronze/raw/Matches.csv


Nombre total de lignes lues       : 238,858
Nombre de partitions Spark créées : 11


---
## 5. Construction de votre Couche Bronze en Delta Lake (`/lakehouse/bronze/`)

### Exercice 5.1 — Convertir vos fichiers bruts en tables Delta Lake Bronze
Dans une architecture **Médaillon**, la couche **Bronze** conserve les données brutes fidèles à la source tout en les stockant au format transactionnel **Delta Lake** enrichi de métadonnées de traçabilité :
- `_ingested_at` : `F.current_timestamp()`
- `_source_file` : `F.input_file_name()`

> **Attention aux noms de colonnes Delta/Parquet :** Si vos fichiers CSV bruts contiennent des espaces ou caractères spéciaux dans les en-têtes, normalisez-les simplement en `snake_case` (ex. `.toDF(*[c.strip().lower().replace(" ", "_") for c in df.columns])`).

Complétez la cellule ci-dessous pour ingérer chacune des tables utiles de votre sujet dans `/lakehouse/bronze/<nom_table>`.

In [13]:

# Définition des fichiers sources
BRONZE_SOURCES: dict[str, str] = {
    "matches": "/lakehouse/bronze/raw/Matches.csv",
    "elo_ratings": "/lakehouse/bronze/raw/EloRatings.csv"
}

assert len(BRONZE_SOURCES) > 0, "Aucune table à ingérer dans BRONZE_SOURCES."

for table_name, raw_path in BRONZE_SOURCES.items():
    bronze_delta_path = f"/lakehouse/bronze/{table_name}"

    # 1. Lecture du fichier CSV depuis HDFS
    df_raw = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(raw_path)
    )

    # 2. Normalisation des noms de colonnes
    df_raw = df_raw.toDF(*[
        c.strip().lower().replace(" ", "_")
        for c in df_raw.columns
    ])

    # 3. Ajout des métadonnées de traçabilité
    df_raw = (
        df_raw
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.input_file_name())
    )

    # 4. Écriture au format Delta Lake
    (
        df_raw.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(bronze_delta_path)
    )

    assert df_raw is not None, f"Erreur lors de la création de {table_name}."

    cnt = spark.read.format("delta").load(bronze_delta_path).count()

    print(f"[BRONZE OK] {bronze_delta_path:<38} -> {cnt:>10,} lignes")


26/11/16 16:06:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[BRONZE OK] /lakehouse/bronze/matches              ->    238,858 lignes


[BRONZE OK] /lakehouse/bronze/elo_ratings          ->    273,972 lignes


In [14]:
# Auto-contrôle de fin de Jour 1 : au moins une table Delta valide dans /lakehouse/bronze/
bronze_root = PathHDFS("/lakehouse/bronze")
bronze_delta_tables = [
    s.getPath().getName()
    for s in fs.listStatus(bronze_root)
    if s.isDirectory() and fs.exists(PathHDFS(s.getPath(), "_delta_log"))
]
assert len(bronze_delta_tables) > 0, "Aucune table Delta détectée dans /lakehouse/bronze/ !"
print("Tables Delta Bronze validées :", bronze_delta_tables)

Tables Delta Bronze validées : ['elo_ratings', 'matches']
